# 00 · Setup — Connections and Schema Creation

This notebook is the starting point of the WWI (Wide World Importers) Data Mart ELT pipeline.

**What this notebook does:**
1. Defines the connection parameters for both databases
2. Tests the connections (operational source and DW)
3. Creates the `bronze`, `silver` and `gold` schemas in the DW database
4. Creates the `gold` layer tables (dimensions and fact tables)
5. Validates the created structure

---
**Medallion Architecture**

```
wwi (OLTP source)               wwi_dw (destination)
────────────────               ────────────────────────────────────
sales.*                ──E──▶  bronze.*   (raw copy)
warehouse.*            ──T──▶  silver.*   (cleansing and transformation)
application.*          ──L──▶  gold.*     (star schema / DW)
purchasing.*
```

**Dimensional Model (Gold):**
```
                dim_date          dim_customer ──── dim_geography
                    │                  │
 dim_delivery ─── fact_sales ──── dim_product
  _method          │
               dim_employee ──── dim_geography

                dim_date          dim_customer
                    │                  │
 dim_delivery ─── fact_orders ─── dim_product
  _method          │
               dim_employee
```

## 1. Dependency Installation

Required libraries:
- `psycopg2-binary` — PostgreSQL driver
- `sqlalchemy` — connection and SQL execution abstraction
- `pandas` — data manipulation
- `python-dotenv` — credential loading from the `.env` file

In [17]:
%pip install psycopg2-binary sqlalchemy pandas python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Connection Parameters

Credentials are loaded from the `.env` file in the same directory.
Copy `.env.example` to `.env` and fill in your values before running.

In [18]:
from dotenv import load_dotenv
import os

load_dotenv()  # loads the .env file in the same directory

# -- Operational source (WWI OLTP) --------------------------------------------
SRC_HOST     = os.getenv("SRC_HOST")
SRC_PORT     = int(os.getenv("SRC_PORT", 5432))
SRC_DB       = os.getenv("SRC_DB")
SRC_USER     = os.getenv("SRC_USER")
SRC_PASSWORD = os.getenv("SRC_PASSWORD")

# -- Data Warehouse (destination) ------------------------------------------------
DWH_HOST     = os.getenv("DWH_HOST")
DWH_PORT     = int(os.getenv("DWH_PORT", 5432))
DWH_DB       = os.getenv("DWH_DB")
DWH_USER     = os.getenv("DWH_USER")
DWH_PASSWORD = os.getenv("DWH_PASSWORD")

# -- Medallion architecture schemas ----------------------------------------
SCHEMAS = ["bronze", "silver", "gold"]

print(f"SRC: {SRC_USER}@{SRC_HOST}:{SRC_PORT}/{SRC_DB}")
print(f"DWH: {DWH_USER}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}")

SRC: dss@postgres2.ipca.pt:5432/wwi
DWH: postgres@localhost:5432/wwi_dw


## 3. SQLAlchemy Engine Creation

In [19]:
from sqlalchemy import create_engine, text

def make_engine(host, port, db, user, password):
    url = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}"
    return create_engine(url)

engine_src = make_engine(SRC_HOST, SRC_PORT, SRC_DB, SRC_USER, SRC_PASSWORD)
engine_dwh = make_engine(DWH_HOST, DWH_PORT, DWH_DB, DWH_USER, DWH_PASSWORD)

print("Engines created successfully.")

Engines created successfully.


## 4. Connection Test

Verifies that both databases are accessible before proceeding.

In [20]:
def test_connection(engine, label):
    try:
        with engine.connect() as conn:
            result = conn.execute(text("SELECT current_database(), version()"))
            db_name, version = result.fetchone()
            print(f"✓ {label}")
            print(f"  Database : {db_name}")
            print(f"  Version  : {version.split(',')[0]}")
    except Exception as e:
        print(f"✗ {label} — ERROR: {e}")
    print()

test_connection(engine_src, f"Operational source ({SRC_DB})")
test_connection(engine_dwh, f"Data Warehouse    ({DWH_DB})")

✓ Operational source (wwi)
  Database : wwi
  Version  : PostgreSQL 17.9 on x86_64-pc-linux-gnu

✓ Data Warehouse    (wwi_dw)
  Database : wwi_dw
  Version  : PostgreSQL 18.3 on x86_64-windows



## 5. Medallion Architecture Schema Creation

The three schemas are created in the DW database. `IF NOT EXISTS` ensures idempotency.

In [21]:
with engine_dwh.begin() as conn:
    for schema in SCHEMAS:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {schema}"))
        print(f"✓ Schema '{schema}' created (or already existed)")

print("\nMedallion architecture schemas ready.")

✓ Schema 'bronze' created (or already existed)
✓ Schema 'silver' created (or already existed)
✓ Schema 'gold' created (or already existed)

Medallion architecture schemas ready.


## 6. Gold Layer Table Creation

The gold layer corresponds to the Data Mart in star schema (Kimball).

**Dimensions with SCD Type 1** (overwrite — no version history):
- `dim_geography`, `dim_customer`, `dim_product`, `dim_employee`, `dim_delivery_method`

**Calendar dimension** (`dim_date`): generated programmatically in notebook `03_gold.ipynb`.

**Outrigger**: `dim_geography` is linked to `dim_customer` and `dim_employee` instead of directly to the fact tables.

**Fact tables**:
- `fact_sales` — granularity: invoice line
- `fact_orders` — granularity: order line

In [22]:
DDL_GOLD = """
-- ─────────────────────────────────────────────
-- DROP existing tables (reverse order by FK)
-- ─────────────────────────────────────────────
DROP TABLE IF EXISTS gold.fact_orders        CASCADE;
DROP TABLE IF EXISTS gold.fact_sales         CASCADE;
DROP TABLE IF EXISTS gold.dim_customer       CASCADE;
DROP TABLE IF EXISTS gold.dim_employee       CASCADE;
DROP TABLE IF EXISTS gold.dim_product        CASCADE;
DROP TABLE IF EXISTS gold.dim_delivery_method CASCADE;
DROP TABLE IF EXISTS gold.dim_geography      CASCADE;
DROP TABLE IF EXISTS gold.dim_date           CASCADE;

-- ─────────────────────────────────────────────
-- DIMENSIONS
-- ─────────────────────────────────────────────

-- dim_date (denormalized calendar, generated programmatically)
CREATE TABLE IF NOT EXISTS gold.dim_date (
    date_key           INT          NOT NULL PRIMARY KEY,
    full_date          DATE         NOT NULL,
    day                SMALLINT     NOT NULL,
    month              SMALLINT     NOT NULL,
    month_name         VARCHAR(15)  NOT NULL,
    quarter            SMALLINT     NOT NULL,
    year               SMALLINT     NOT NULL,
    week_of_year       SMALLINT     NOT NULL,
    day_of_week        SMALLINT     NOT NULL,
    day_name           VARCHAR(10)  NOT NULL,
    is_weekend         BOOLEAN      NOT NULL,
    is_holiday         BOOLEAN      NOT NULL DEFAULT FALSE
);

-- dim_geography (outrigger — linked via dim_customer and dim_employee)
CREATE TABLE IF NOT EXISTS gold.dim_geography (
    geography_key       SERIAL       PRIMARY KEY,
    city_name           VARCHAR(100) NOT NULL,
    state_province_name VARCHAR(100),
    country_name        VARCHAR(100)
);

-- dim_customer (SCD Type 2: customer_category_name, buying_group_name, is_on_credit_hold)
CREATE TABLE IF NOT EXISTS gold.dim_customer (
    customer_key           SERIAL       PRIMARY KEY,
    customer_id            INT          NOT NULL,
    customer_name          VARCHAR(100),
    customer_category_name VARCHAR(50),
    buying_group_name      VARCHAR(50),
    bill_to_customer_name  VARCHAR(100),
    phone_number           VARCHAR(100),
    is_on_credit_hold      BOOLEAN,
    geography_key          INT          REFERENCES gold.dim_geography(geography_key),
    start_date             DATE         NOT NULL DEFAULT CURRENT_DATE,
    end_date               DATE         NOT NULL DEFAULT '9999-12-31',
    is_active              BOOLEAN      NOT NULL DEFAULT TRUE
);

-- dim_product (SCD Type 2: brand, size, tax_rate, lead_time_days)
CREATE TABLE IF NOT EXISTS gold.dim_product (
    product_key        SERIAL         PRIMARY KEY,
    stock_item_id      INT            NOT NULL,
    stock_item_name    VARCHAR(100),
    color_name         VARCHAR(30),
    unit_package_name  VARCHAR(50),
    outer_package_name VARCHAR(50),
    brand              VARCHAR(50),
    size               VARCHAR(30),
    lead_time_days     INT,
    quantity_per_outer INT,
    is_chiller_stock   BOOLEAN,
    tax_rate           NUMERIC(5,2),
    start_date         DATE           NOT NULL DEFAULT CURRENT_DATE,
    end_date           DATE           NOT NULL DEFAULT '9999-12-31',
    is_active          BOOLEAN        NOT NULL DEFAULT TRUE
);

-- dim_employee (SCD Type 2: is_salesperson)
CREATE TABLE IF NOT EXISTS gold.dim_employee (
    employee_key   SERIAL       PRIMARY KEY,
    person_id      INT          NOT NULL,
    full_name      VARCHAR(100),
    preferred_name VARCHAR(50),
    is_salesperson BOOLEAN,
    geography_key  INT          REFERENCES gold.dim_geography(geography_key),
    start_date     DATE         NOT NULL DEFAULT CURRENT_DATE,
    end_date       DATE         NOT NULL DEFAULT '9999-12-31',
    is_active      BOOLEAN      NOT NULL DEFAULT TRUE
);

-- dim_delivery_method (SCD Type 1)
CREATE TABLE IF NOT EXISTS gold.dim_delivery_method (
    delivery_method_key  SERIAL      PRIMARY KEY,
    delivery_method_id   INT         NOT NULL UNIQUE,
    delivery_method_name VARCHAR(50)
);

-- ─────────────────────────────────────────────
-- PARTIAL INDEXES (ensures 1 current record per business key)
-- ─────────────────────────────────────────────
CREATE UNIQUE INDEX IF NOT EXISTS ix_dim_customer_current
    ON gold.dim_customer (customer_id) WHERE is_active = TRUE;

CREATE UNIQUE INDEX IF NOT EXISTS ix_dim_product_current
    ON gold.dim_product (stock_item_id) WHERE is_active = TRUE;

CREATE UNIQUE INDEX IF NOT EXISTS ix_dim_employee_current
    ON gold.dim_employee (person_id) WHERE is_active = TRUE;

-- ─────────────────────────────────────────────
-- FACT TABLES
-- ─────────────────────────────────────────────

-- fact_sales (granularity: invoice line)
CREATE TABLE IF NOT EXISTS gold.fact_sales (
    customer_key              INT          REFERENCES gold.dim_customer(customer_key),
    date_key                  INT          REFERENCES gold.dim_date(date_key),
    delivery_method_key       INT          REFERENCES gold.dim_delivery_method(delivery_method_key),
    product_key               INT          REFERENCES gold.dim_product(product_key),
    salesperson_employee_key  INT          REFERENCES gold.dim_employee(employee_key),
    invoice_id                INT,
    order_id                  INT,
    quantity                  INT,
    unit_price                NUMERIC(18,2),
    tax_amount                NUMERIC(18,2),
    line_total_excl_tax       NUMERIC(18,2)
);

-- fact_orders (granularity: order line)
CREATE TABLE IF NOT EXISTS gold.fact_orders (
    customer_key              INT          REFERENCES gold.dim_customer(customer_key),
    order_date_key            INT          REFERENCES gold.dim_date(date_key),
    product_key               INT          REFERENCES gold.dim_product(product_key),
    expected_delivery_date_key INT         REFERENCES gold.dim_date(date_key),
    salesperson_employee_key  INT          REFERENCES gold.dim_employee(employee_key),
    order_id                  INT,
    ordered_quantity          INT,
    picked_quantity           INT,
    backordered_quantity      INT
);

-- ─────────────────────────────────────────────
-- INDEXES (performance)
-- ─────────────────────────────────────────────

CREATE INDEX IF NOT EXISTS ix_fact_sales_date    ON gold.fact_sales(date_key);
CREATE INDEX IF NOT EXISTS ix_fact_sales_cust    ON gold.fact_sales(customer_key);
CREATE INDEX IF NOT EXISTS ix_fact_sales_prod    ON gold.fact_sales(product_key);
CREATE INDEX IF NOT EXISTS ix_fact_sales_emp     ON gold.fact_sales(salesperson_employee_key);
CREATE INDEX IF NOT EXISTS ix_fact_sales_dm      ON gold.fact_sales(delivery_method_key);

CREATE INDEX IF NOT EXISTS ix_fact_orders_date   ON gold.fact_orders(order_date_key);
CREATE INDEX IF NOT EXISTS ix_fact_orders_exp    ON gold.fact_orders(expected_delivery_date_key);
CREATE INDEX IF NOT EXISTS ix_fact_orders_cust   ON gold.fact_orders(customer_key);
CREATE INDEX IF NOT EXISTS ix_fact_orders_prod   ON gold.fact_orders(product_key);
CREATE INDEX IF NOT EXISTS ix_fact_orders_emp    ON gold.fact_orders(salesperson_employee_key);
"""

with engine_dwh.begin() as conn:
    conn.execute(text(DDL_GOLD))

print("✓ Gold layer tables created successfully.")

✓ Gold layer tables created successfully.


## 7. Validation — Inventory of created schemas and tables

In [23]:
import pandas as pd

query = """
    SELECT table_schema AS schema, table_name AS table_name
    FROM   information_schema.tables
    WHERE  table_schema IN ('bronze', 'silver', 'gold')
    ORDER  BY table_schema, table_name
"""

with engine_dwh.connect() as conn:
    df = pd.read_sql(text(query), conn)

print("Objects created in the DW:")
print(df.to_string(index=False))

Objects created in the DW:
schema           table_name
bronze        _load_control
bronze         buyinggroups
bronze               cities
bronze               colors
bronze            countries
bronze   customercategories
bronze            customers
bronze      deliverymethods
bronze         invoicelines
bronze             invoices
bronze           orderlines
bronze               orders
bronze         packagetypes
bronze               people
bronze       stateprovinces
bronze          stockgroups
bronze           stockitems
  gold         dim_customer
  gold             dim_date
  gold  dim_delivery_method
  gold         dim_employee
  gold        dim_geography
  gold          dim_product
  gold          fact_orders
  gold           fact_sales
silver        stg_customers
silver stg_delivery_methods
silver        stg_employees
silver        stg_geography
silver    stg_invoice_lines
silver         stg_invoices
silver      stg_order_lines
silver           stg_orders
silver         stg_pr

## 8. Validation — Available tables in the WWI source

In [24]:
query_src = """
    SELECT table_schema AS schema,
           table_name   AS table_name
    FROM   information_schema.tables
    WHERE  table_schema NOT IN ('pg_catalog', 'information_schema')
      AND  table_type = 'BASE TABLE'
    ORDER  BY table_schema, table_name
"""

with engine_src.connect() as conn:
    df_src = pd.read_sql(text(query_src), conn)

print(f"Available tables in the source ({SRC_DB}):")
print(df_src.to_string(index=False))

Available tables in the source (wwi):
schema           table_name
public         buyinggroups
public               cities
public               colors
public            countries
public   customercategories
public            customers
public customertransactions
public      deliverymethods
public         invoicelines
public             invoices
public           orderlines
public               orders
public         packagetypes
public       paymentmethods
public               people
public         specialdeals
public       stateprovinces
public          stockgroups
public           stockitems
public     transactiontypes


## 9. Summary

If the previous cells ran without errors, the environment is ready:

| Step | Status |
|---|---|
| Connection to WWI source | ✓ |
| Connection to DW | ✓ |
| Schema `bronze` created | ✓ |
| Schema `silver` created | ✓ |
| Schema `gold` created | ✓ |
| Gold tables created | ✓ |

> **⚠ Warning:** Check cell 8 to confirm the exact names of schemas
> and tables available in the WWI source. Update the file `01_bronze.ipynb`
> if table or schema names differ from expected.

**Next step:** run notebook `01_bronze.ipynb`.